# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What state received the most CMS payments in 2025?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-10-01 16:58:24 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC), Pennsylvania Open Data Portal (data.pa.gov), datHere CKAN Portal, Verikan Fair Store |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-10-01T16:58:24.587643")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'CMS payments by state 2025':

1. **25% or Greater Slope** (13.3% match)
   Resource: CSV
   Resource ID: `3ba94030-bda4-47e5-9e8d-a5ae1aef569c`
   Dataset ID: `25-or-greater-slope`
   Format: CSV | Rows: 1,714 | Cols: 12
   Tags: geometric_shapes, area, length, spatial_data, administrative_data, system_generated_id, record_creation, shape_attributes
   This dataset contains 1714 records, each with a unique system-generated identifier, `_id`. It includes observational or administrative data, with fields such as `objectid_1`, `created_user`, `created
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'CMS payments by state 2025', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `CMS payments`

**Result preview:**
```
Found 0 datasets matching 'CMS payments'

```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'CMS payments', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Search for Datasets

**Search query:** `CMS payments`

**Result preview:**
```
Found 4 datasets matching 'CMS payments':

1. **Center for Medicare and Medicaid Open Payments Open Data Portal Federal**
   Dataset ID: `ke8j-cm5k`  (relevance 0.83)
   Every year, CMS will update the Open Payments data at least once after its initial publication. The refreshed data will include updates to data disputes and other data corrections made since the initial publication of this data documenting payments o
   Keywords: medicare, medicaid, payment, health, provider
   Modified: 2020-07-29 | Distributions: 1 (0 tabular)

2. **Centers for Medicare and Medicaid Services Data Portal Fede
```


In [ ]:
# Step 3: Search for Datasets

# Search the DCAT catalog (fetched once, searched locally)
import requests

CATALOG_URL = 'https://data.pa.gov/data.json'
QUERY = 'CMS payments'

catalog = requests.get(CATALOG_URL, timeout=60).json()
datasets = catalog.get('dataset', [])
print(f'Catalog: {len(datasets)} datasets')

def short_id(ds):
    """Portal dataset ID = last path segment of the DCAT identifier."""
    ident = ds.get('identifier') or ''
    return ident.rstrip('/').rsplit('/', 1)[-1]

terms = [t for t in QUERY.lower().split() if t]

def score(ds):
    title = (ds.get('title') or '').lower()
    desc = (ds.get('description') or '').lower()
    kws = ' '.join(ds.get('keyword') or []).lower()
    return sum(3 * (t in title) + 2 * (t in kws) + (t in desc) for t in terms)

matches = [d for d in datasets if score(d) > 0]
matches.sort(key=score, reverse=True)

for ds in matches[:10]:
    print(f"{short_id(ds):14} {ds.get('title', '')}")


## Step 4: Search for Datasets

**Search query:** `Medicare Medicaid payments by state`

**Result preview:**
```
Found 0 datasets matching 'Medicare Medicaid payments by state'

```


In [ ]:
# Step 4: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'Medicare Medicaid payments by state', "rows": 10}
resp = requests.get("https://data.dathere.com/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 5: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'Medicare Medicaid Centers for Medicare and Medicaid Services payments':

1. **City Revenues and Expenses** (16.4% match)
   Resource: City Wide Revenues and Expenses
   Resource ID: `f61f6e8c-7b93-4df3-9935-4937899901c7`
   Dataset ID: `city-revenues-and-expenses`
   Format: CSV | Rows: 617,753 | Cols: 13
   Tags: financial_transactions, expenses, revenues, fund_management, department_accounting, cost_centers, object_accounts, general_ledger, fiscal_year, budgeting
   This dataset contains financial transaction data, primarily detailing expenses an
```


In [ ]:
# Step 5: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'Medicare Medicaid Centers for Medicare and Medicaid Services payments', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 6: Dataset Details (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Dataset:** `ke8j-cm5k`

**Result preview:**
```
HTTP 404: {"help": "https://data.wprdc.org/api/3/action/help_show?name=package_show", "error": {"__type": "Not Found Error", "message": "Not found"}, "success": false}
```


In [ ]:
# Step 6: Dataset Details — failed exploratory attempt, kept for provenance
# # Get dataset details
# resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'ke8j-cm5k'})
# ds = resp.json()["result"]
# 
# print(f"Title: {ds['title']}")
# print(f"Description: {ds.get('notes', 'N/A')[:200]}")
# print(f"\nResources:")
# for r in ds.get("resources", []):
#     act = "DataStore" if r.get("datastore_active") else "File"
#     print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")

## Step 7: Dataset Details (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Dataset:** `3swi-hqsg`

**Result preview:**
```
HTTP 404: {"help": "https://data.wprdc.org/api/3/action/help_show?name=package_show", "error": {"__type": "Not Found Error", "message": "Not found"}, "success": false}
```


In [ ]:
# Step 7: Dataset Details — failed exploratory attempt, kept for provenance
# # Get dataset details
# resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": '3swi-hqsg'})
# ds = resp.json()["result"]
# 
# print(f"Title: {ds['title']}")
# print(f"Description: {ds.get('notes', 'N/A')[:200]}")
# print(f"\nResources:")
# for r in ds.get("resources", []):
#     act = "DataStore" if r.get("datastore_active") else "File"
#     print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")

## Step 8: Dataset Details (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Dataset:** `ke8j-cm5k`

**Result preview:**
```
HTTP 404: {"help": "https://data.dathere.com/api/3/action/help_show?name=package_show", "error": {"__type": "Not Found Error", "message": "Not found"}, "success": false}
```


In [ ]:
# Step 8: Dataset Details — failed exploratory attempt, kept for provenance
# # Get dataset details
# resp = requests.get("https://data.dathere.com/api/3/action/package_show", params={"id": 'ke8j-cm5k'})
# ds = resp.json()["result"]
# 
# print(f"Title: {ds['title']}")
# print(f"Description: {ds.get('notes', 'N/A')[:200]}")
# print(f"\nResources:")
# for r in ds.get("resources", []):
#     act = "DataStore" if r.get("datastore_active") else "File"
#     print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")

## Step 9: Dataset Details (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Dataset:** `3swi-hqsg`

**Result preview:**
```
HTTP 404: {"help": "https://data.dathere.com/api/3/action/help_show?name=package_show", "error": {"__type": "Not Found Error", "message": "Not found"}, "success": false}
```


In [ ]:
# Step 9: Dataset Details — failed exploratory attempt, kept for provenance
# # Get dataset details
# resp = requests.get("https://data.dathere.com/api/3/action/package_show", params={"id": '3swi-hqsg'})
# ds = resp.json()["result"]
# 
# print(f"Title: {ds['title']}")
# print(f"Description: {ds.get('notes', 'N/A')[:200]}")
# print(f"\nResources:")
# for r in ds.get("resources", []):
#     act = "DataStore" if r.get("datastore_active") else "File"
#     print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")

## Step 10: Dataset Details (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Dataset:** `ke8j-cm5k`

**Result preview:**
```
HTTP 404: {"help": "https://34-56-220-178.sslip.io/api/3/action/help_show?name=package_show", "error": {"__type": "Not Found Error", "message": "Not found"}, "success": false}
```


In [ ]:
# Step 10: Dataset Details — failed exploratory attempt, kept for provenance
# # Get dataset details
# resp = requests.get("https://34-56-220-178.sslip.io/api/3/action/package_show", params={"id": 'ke8j-cm5k'})
# ds = resp.json()["result"]
# 
# print(f"Title: {ds['title']}")
# print(f"Description: {ds.get('notes', 'N/A')[:200]}")
# print(f"\nResources:")
# for r in ds.get("resources", []):
#     act = "DataStore" if r.get("datastore_active") else "File"
#     print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")

## Step 11: Search for Datasets

**Search query:** `CMS payments`

**Result preview:**
```
Found 2 datasets matching 'CMS payments'

1. **Center for Medicare and Medicaid Open Payments Open Data Portal Federal**
   ID: `center-for-medicare-and-medicaid-open-payments-open-data-portal-federal-ke8j-cm5k`
   Every year, CMS will update the Open Payments data at least once after its initial publication. The refreshed data will include updates to data disputes and other data corrections 
   - HTML (HTML) ID: `07782ea0-6bb1-5de1-b389-c99e84815197`

2. **2025 Program Year**
   ID: `2025-program-year`
   Every year, CMS publishes complete datasets that consolidate the information submitted b
```


In [ ]:
# Step 11: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'CMS payments', "rows": 10}
resp = requests.get("https://34-56-220-178.sslip.io/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 12: Search for Datasets

**Search query:** `CMS Open Payments`

**Result preview:**
```
Found 1 datasets matching 'CMS Open Payments'

1. **2025 Program Year**
   ID: `2025-program-year`
   Every year, CMS publishes complete datasets that consolidate the information submitted by reporting entities for active years of Open Payments data. You can download the full datas
   - 2025 Program Year (ZIP) [DataStore] ID: `c15728f5-bb96-43a6-9645-39c5f8aff550`
   - General payments 2025: totals by company and payment type (CSV) [DataStore] ID: `23cdb97f-61c4-4813-9028-957ae3300c76`
   - General payments 2025: totals by recipient state and specialty (CSV) [DataStore] ID: `7c0d3693-9458-4ee4
```


In [ ]:
# Step 12: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'CMS Open Payments', "rows": 10}
resp = requests.get("https://data.dathere.com/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 13: Search for Datasets

**Search query:** `CMS payments`

**Result preview:**
```
Found 4 datasets matching 'CMS payments':

1. **Center for Medicare and Medicaid Open Payments Open Data Portal Federal**
   Dataset ID: `ke8j-cm5k`  (relevance 0.83)
   Every year, CMS will update the Open Payments data at least once after its initial publication. The refreshed data will include updates to data disputes and other data corrections made since the initial publication of this data documenting payments o
   Keywords: medicare, medicaid, payment, health, provider
   Modified: 2020-07-29 | Distributions: 1 (0 tabular)

2. **Centers for Medicare and Medicaid Services Data Portal Fede
```


In [ ]:
# Step 13: Search for Datasets

# Search the DCAT catalog (fetched once, searched locally)
import requests

CATALOG_URL = 'https://data.pa.gov/data.json'
QUERY = 'CMS payments'

catalog = requests.get(CATALOG_URL, timeout=60).json()
datasets = catalog.get('dataset', [])
print(f'Catalog: {len(datasets)} datasets')

def short_id(ds):
    """Portal dataset ID = last path segment of the DCAT identifier."""
    ident = ds.get('identifier') or ''
    return ident.rstrip('/').rsplit('/', 1)[-1]

terms = [t for t in QUERY.lower().split() if t]

def score(ds):
    title = (ds.get('title') or '').lower()
    desc = (ds.get('description') or '').lower()
    kws = ' '.join(ds.get('keyword') or []).lower()
    return sum(3 * (t in title) + 2 * (t in kws) + (t in desc) for t in terms)

matches = [d for d in datasets if score(d) > 0]
matches.sort(key=score, reverse=True)

for ds in matches[:10]:
    print(f"{short_id(ds):14} {ds.get('title', '')}")


## Step 14: Dataset Details

**Dataset:** `2025-program-year`

**Result preview:**
```
# 2025 Program Year

Every year, CMS publishes complete datasets that consolidate the information submitted by reporting entities for active years of Open Payments data. You can download the full datasets below or use the Open Payments Search Tool to refine and filter views of this data. Datasets from previous program years are available on the Archived Datasets page.

Organization: CMS
Modified: 2026-10-01
Tags: cms, government, health statistics, industry payments, medical devices, open payments, pharmaceutical, physician payments, sunshine act, teaching hospitals

Resources (18):
  1. 2025 
```


In [ ]:
# Step 14: Dataset Details

# Get dataset details
resp = requests.get("https://data.dathere.com/api/3/action/package_show", params={"id": '2025-program-year'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 15: Dataset Details

**Dataset:** `2025-program-year`

**Result preview:**
```
# 2025 Program Year

Every year, CMS publishes complete datasets that consolidate the information submitted by reporting entities for active years of Open Payments data. You can download the full datasets below or use the Open Payments Search Tool to refine and filter views of this data. Datasets from previous program years are available on the Archived Datasets page.

Organization: CMS
Modified: 2026-10-01
Tags: cms, government, health statistics
Mirrored from: datHere CKAN Portal (portal_id `ckan`)

Resources (1):
  1. 2025 Program Year
     ID: `c15728f5-bb96-43a6-9645-39c5f8aff550`
     Fo
```


In [ ]:
# Step 15: Dataset Details

# Get dataset details
resp = requests.get("https://34-56-220-178.sslip.io/api/3/action/package_show", params={"id": '2025-program-year'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 16: Load Data from Resource (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**Resource ID:** `e1f71a48-8638-4564-87cd-6a0c925f7450`
**Limit:** 10

**Result preview:**
```
HTTP 404: {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search", "error": {"__type": "Not Found Error", "message": "Not found: Resource \"e1f71a48-8638-4564-87cd-6a0c925f7450\" was not found."}, "success": false}
```


In [ ]:
# Step 16: Load Data from Resource — failed exploratory attempt, kept for provenance
# # Load resource data
# import pandas as pd
# 
# params = {"resource_id": 'e1f71a48-8638-4564-87cd-6a0c925f7450', "limit": 10, "sort": 'total_amount desc'}
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
# print(f"Total available: {result['total']:,}")
# print(f"\nColumns: {', '.join(df.columns.tolist())}")
# df.head(10)

## Step 17: Load Data from Resource

**Resource ID:** `e1f71a48-8638-4564-87cd-6a0c925f7450`
**Limit:** 10

**Result preview:**
```
Resource: e1f71a48-8638-4564-87cd-6a0c925f7450
Total records: 59
Loaded: 10
Fields (7): rank, recipient_state, records, total_usd, recipients, largest_paying_company, largest_paying_company_usd

Sample (10 rows):

 rank recipient_state  records    total_usd  recipients            largest_paying_company  largest_paying_company_usd
    1              PA   689644 713306030.33       47651                       BioNTech SE                535843017.28
    2              CA  1450114 423777654.79      102664                   Novo Nordisk AS                 55523760.80
    3              NY  1089903 2
```


In [ ]:
# Step 17: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": 'e1f71a48-8638-4564-87cd-6a0c925f7450', "limit": 10}
resp = requests.post("https://data.dathere.com/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **## Answer

**Pennsylvania received the most CMS Open Payments in 2025**, with a total of **$713,306,030.33** across 689,644 payment records to 47,651 recipients. The single largest contributor was **BioNTech SE**, which paid Pennsylvania recipients $535,843,017.28 (likely reflecting large payments to University of Pennsylvania-affiliated researchers tied to mRNA vaccine technology).

### Top 5 States by Total CMS General Payments (2025 Program Year)
| Rank | State | Total Payments (USD) | # Records | # Recipients | Largest Paying Company |
|------|-------|----------------------|-----------|---------------|--------------------------|
| 1 | PA | $713,306,030.33 | 689,644 | 47,651 | BioNTech SE ($535.8M) |
| 2 | CA | $423,777,654.79 | 1,450,114 | 102,664 | Novo Nordisk AS ($55.5M) |
| 3 | NY | $260,029,754.53 | 1,089,903 | 74,524 | Boston Scientific Corp ($18.8M) |
| 4 | TX | $241,193,868.70 | 1,518,471 | 90,998 | AbbVie Inc. ($27.2M) |
| 5 | FL | $231,575,568.52 | 1,436,414 | 89,041 | AbbVie Inc. ($11.7M) |

**Source:** "General payments 2025: national totals by recipient state (ranked)" — part of the CMS "2025 Program Year" Open Payments dataset (derived from file OP_DTL_GNRL_PGYR2025_P06302026_06032026.csv, 16,131,856 total general payment records), available via the datHere CKAN Portal (`ckan`).

**Caveat:** This covers *general payments* (e.g., consulting fees, gifts, speaking fees, royalties) under the CMS Open Payments ("Sunshine Act") program — it does not include separate research payments or ownership/investment interest datasets, which are broken out by company rather than by state in this portal's holdings. Pennsylvania's outsized total is heavily skewed by one very large BioNTech payment category, so excluding that single company, California would lead among the remaining states.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-10-01

**[2]** Pennsylvania Open Data Portal (data.pa.gov)
- Dataset: Pennsylvania Open Data Portal (data.pa.gov)
- URL: [https://data.pa.gov](https://data.pa.gov)
- Accessed: 2026-10-01

**[3]** datHere CKAN Portal
- Dataset: datHere CKAN Portal
- URL: [https://data.dathere.com](https://data.dathere.com)
- Accessed: 2026-10-01

**[4]** Verikan Fair Store
- Dataset: Verikan Fair Store
- URL: [https://34-56-220-178.sslip.io](https://34-56-220-178.sslip.io)
- Accessed: 2026-10-01

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-10-01 16:58:24
- **Query**: What state received the most CMS payments in 2025?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
